# Drug Discovery Compound Filtering Pipeline

This notebook implements a **progressive, memory-efficient filtering pipeline** for large compound libraries (~600k SMILES). Filters are ordered from computationally cheapest to most expensive, with a hard compound limiter applied before any memory-intensive step.

## Pipeline Overview

| Stage | Filter | Rationale | Complexity |
|-------|--------|-----------|------------|
| 0 | SMILES validity | Remove unparseable entries | O(n), string-level |
| 1 | Lipinski Ro5 | Oral bioavailability (MW, logP, HBD, HBA) | O(n), fast descriptors |
| 2 | Veber rules | Oral bioavailability (TPSA, rotatable bonds) | O(n), fast descriptors |
| 3 | Egan/Lead-like filters | Lead optimization space | O(n), fast descriptors |
| 4 | PAINS / Structural alerts | Promiscuous binders, reactive groups | O(n), substructure |
| 5 | **Compound limiter (n=1000)** | Cap compounds before expensive steps | — |
| 6 | ADMET properties | Absorption, distribution, metabolism | O(n), heavier calc |
| 7 | 3D conformer generation | Pre-docking quality check | O(n), expensive |

## References
- Lipinski et al. (2001) *Adv. Drug Deliv. Rev.* 46:3–26
- Veber et al. (2002) *J. Med. Chem.* 45:2615–2623
- Egan et al. (2000) *J. Med. Chem.* 43:3867–3877
- Baell & Holloway (2010) *J. Med. Chem.* 53:2719–2740 (PAINS)
- Brenk et al. (2008) *ChemMedChem* 3:435–444 (structural alerts)
- Oprea (2000) *J. Comput.-Aided Mol. Des.* 14:251–264 (lead-like)

## 0. Environment Setup

In [8]:
# Install dependencies if needed
# !pip install rdkit pandas numpy tqdm

import warnings
warnings.filterwarnings('ignore')

import os
import gc
import time
import logging
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from rdkit import Chem
from rdkit import RDLogger
from rdkit.Chem import Descriptors, rdMolDescriptors, FilterCatalog
from rdkit.Chem.FilterCatalog import FilterCatalogParams
from rdkit.Chem import AllChem

# Suppress RDKit warnings for cleaner output
RDLogger.DisableLog('rdApp.*')

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%H:%M:%S'
)
log = logging.getLogger(__name__)

print('RDKit and dependencies loaded successfully.')

RDKit and dependencies loaded successfully.


## 1. Configuration

In [10]:
# ============================================================
# USER CONFIGURATION — edit these parameters
# ============================================================

INPUT_CSV       = r"C:\Users\Christian\OFFTOX\data\inactive_cpds.csv"   # Path to your input CSV
SMILES_COL      = 'SMILES'          # Column name containing SMILES
ID_COL          = None              # Column name for compound IDs (set None if absent)
OUTPUT_DIR      = Path('filtered_output')
CHUNK_SIZE      = 5_000             # Rows per chunk for streaming ingestion

# Hard limiter applied BEFORE memory-intensive steps (Stage 5)
COMPOUND_LIMIT  = 10000

# ── Lipinski Ro5 (Stage 1) ──────────────────────────────────
RO5_MW_MAX      = 500               # Molecular weight (Da)
RO5_LOGP_MAX    = 5.0               # Wildman-Crippen logP
RO5_HBD_MAX     = 5                 # H-bond donors
RO5_HBA_MAX     = 10                # H-bond acceptors
RO5_VIOLATIONS  = 1                 # Max allowed violations (1 = strict Ro5, 2 = relaxed)

# ── Veber (Stage 2) ─────────────────────────────────────────
VEBER_TPSA_MAX  = 140               # Topological polar surface area (Å²)
VEBER_ROTB_MAX  = 10                # Rotatable bond count

# ── Lead-like / Egan (Stage 3) ──────────────────────────────
LEAD_MW_MAX     = 450               # Lead-like MW ceiling (Oprea 2000)
LEAD_LOGP_MAX   = 4.0               # Lead-like logP ceiling
LEAD_RINGS_MAX  = 4                 # Max ring count
EGAN_TPSA_MAX   = 131.6             # Egan TPSA threshold
EGAN_LOGP_MAX   = 5.88              # Egan logP threshold

# ── ADMET (Stage 6) ─────────────────────────────────────────
ADMET_MW_MIN    = 150               # Remove fragment-like compounds
ADMET_QED_MIN   = 0.3               # Quantitative estimate of drug-likeness
ADMET_STEREO_MAX = 3                # Max stereocentres (complexity proxy)

OUTPUT_DIR.mkdir(exist_ok=True)
print(f'Output directory: {OUTPUT_DIR.resolve()}')
print(f'Compound limiter set to: {COMPOUND_LIMIT:,}')

Output directory: C:\Users\Christian\OFFTOX\filtered_output
Compound limiter set to: 10,000


## 2. Helper Functions

In [3]:
def smiles_to_mol(smi):
    """Parse SMILES to RDKit Mol; return None on failure."""
    if not isinstance(smi, str) or not smi.strip():
        return None
    try:
        mol = Chem.MolFromSmiles(smi.strip())
        return mol  # None if invalid
    except Exception:
        return None


def compute_fast_descriptors(mol):
    """
    Compute lightweight 2D descriptors in a single pass.
    Returns a dict; all values are floats/ints.
    """
    mw   = Descriptors.ExactMolWt(mol)
    logp = Descriptors.MolLogP(mol)
    hbd  = rdMolDescriptors.CalcNumHBD(mol)
    hba  = rdMolDescriptors.CalcNumHBA(mol)
    tpsa = Descriptors.TPSA(mol)
    rotb = rdMolDescriptors.CalcNumRotatableBonds(mol)
    rings = rdMolDescriptors.CalcNumRings(mol)
    heavy = mol.GetNumHeavyAtoms()
    return dict(mw=mw, logp=logp, hbd=hbd, hba=hba,
                tpsa=tpsa, rotb=rotb, rings=rings, heavy=heavy)


def lipinski_ro5(d, max_violations=RO5_VIOLATIONS):
    """
    Lipinski Rule of Five (2001). 
    Default: 1 violation allowed (common practice for CNS/natural product scaffolds).
    """
    violations = sum([
        d['mw']   > RO5_MW_MAX,
        d['logp'] > RO5_LOGP_MAX,
        d['hbd']  > RO5_HBD_MAX,
        d['hba']  > RO5_HBA_MAX,
    ])
    return violations <= max_violations


def veber_filter(d):
    """Veber et al. (2002): oral bioavailability in rats. Both criteria must pass."""
    return d['tpsa'] <= VEBER_TPSA_MAX and d['rotb'] <= VEBER_ROTB_MAX


def lead_like_filter(d):
    """
    Lead-like space (Oprea 2000) + Egan absorption model (2000).
    Compounds that satisfy either criterion are retained to maximise scaffold coverage.
    """
    oprea = (d['mw'] <= LEAD_MW_MAX and
             d['logp'] <= LEAD_LOGP_MAX and
             d['rings'] <= LEAD_RINGS_MAX)
    egan  = (d['tpsa'] <= EGAN_TPSA_MAX and
             d['logp'] <= EGAN_LOGP_MAX)
    return oprea or egan


def compute_qed(mol):
    """Quantitative Estimate of Drug-likeness (Bickerton et al. 2012, Nat Chem)."""
    from rdkit.Chem import QED
    try:
        return QED.qed(mol)
    except Exception:
        return 0.0


def count_stereocentres(mol):
    """Count defined stereocentres (complexity / synthetic accessibility proxy)."""
    try:
        si = Chem.FindMolChiralCenters(mol, includeUnassigned=False)
        return len(si)
    except Exception:
        si = rdMolDescriptors.CalcNumAtomStereoCenters(mol)
        return si


def report_stage(stage_name, n_before, n_after):
    removed  = n_before - n_after
    pct_kept = 100 * n_after / max(n_before, 1)
    log.info(f'[{stage_name}] {n_before:,} → {n_after:,}  '
             f'(removed {removed:,}, kept {pct_kept:.1f}%)')

print('Helper functions defined.')

Helper functions defined.


## 3. Stage 0 — Streaming Ingestion & SMILES Validity

**Rationale:** Invalid SMILES strings cannot be parsed into molecular graphs and must be removed before any descriptor calculation. Streaming ingestion via `pandas.read_csv` with `chunksize` avoids loading 600k rows into RAM simultaneously.

Only canonical SMILES and computed descriptors are retained in memory — the original DataFrame is discarded after each chunk.

In [4]:
t0 = time.time()

valid_records = []  # list of dicts; one per valid compound
n_total   = 0
n_invalid = 0

log.info(f'Streaming {INPUT_CSV} in chunks of {CHUNK_SIZE:,} rows...')

reader = pd.read_csv(
    INPUT_CSV,
    usecols=[c for c in [ID_COL, SMILES_COL] if c],  # only load needed columns
    chunksize=CHUNK_SIZE,
    dtype=str,           # read everything as string; avoids dtype coercion issues
    low_memory=True,
)

for chunk_idx, chunk in enumerate(tqdm(reader, desc='Ingesting chunks', unit='chunk')):
    chunk = chunk.dropna(subset=[SMILES_COL])
    n_total += len(chunk)

    for row in chunk.itertuples(index=False):
        smi = getattr(row, SMILES_COL, None)
        cid = getattr(row, ID_COL, f'cpd_{n_total}') if ID_COL else f'cpd_{n_total}'

        mol = smiles_to_mol(smi)
        if mol is None:
            n_invalid += 1
            continue

        # Compute fast descriptors immediately; discard Mol object to save RAM
        desc = compute_fast_descriptors(mol)
        canon_smi = Chem.MolToSmiles(mol)  # canonical form

        valid_records.append({'id': cid, 'smiles': canon_smi, **desc})

    del chunk  # explicit cleanup
    gc.collect()

df = pd.DataFrame(valid_records)
del valid_records
gc.collect()

report_stage('Stage 0 – Validity', n_total, len(df))
log.info(f'  Invalid SMILES discarded: {n_invalid:,}')
log.info(f'  Elapsed: {time.time()-t0:.1f}s')
df.head(3)

20:16:57 [INFO] Streaming C:\Users\Christian\OFFTOX\data\inactive_cpds.csv in chunks of 5,000 rows...
Ingesting chunks: 130chunk [07:09,  3.30s/chunk]
20:24:07 [INFO] [Stage 0 – Validity] 645,909 → 645,779  (removed 130, kept 100.0%)
20:24:07 [INFO]   Invalid SMILES discarded: 130
20:24:07 [INFO]   Elapsed: 430.3s


,id,smiles,mw,logp,hbd,hba,tpsa,rotb,rings,heavy
0,cpd_5000,O=C(Cc1ccccc1Cl)NNC(=O)c1ccccc1OC(F)F,354.058276,2.94510,2,3,67.43,5,2,24
1,cpd_5000,COc1ccc(NC(=O)CSc2cc3nc(C)cc(C)n3n2)cc1,342.115047,3.08554,1,5,68.52,5,3,24
2,cpd_5000,Cc1ccc(Cl)c(NC(=O)CNC(=O)CN2C(=O)NC3(CCCCC3)C2...,406.140783,1.95782,3,4,107.61,5,3,28


## 4. Stage 1 — Lipinski Rule of Five

**Rationale:** Lipinski's Ro5 encodes physicochemical property boundaries derived from orally administered drugs that reached Phase II clinical trials (Lipinski et al., 2001). MW ≤ 500 Da, clogP ≤ 5, HBD ≤ 5, HBA ≤ 10. One violation is commonly permitted to accommodate biologically active natural product scaffolds.

All descriptors were pre-computed during streaming — this stage is a pure DataFrame boolean mask (vectorised, O(n)).

In [5]:
n_before = len(df)

# Compute per-compound violation counts vectorially
df['ro5_viol'] = (
    (df['mw']   > RO5_MW_MAX).astype(int) +
    (df['logp'] > RO5_LOGP_MAX).astype(int) +
    (df['hbd']  > RO5_HBD_MAX).astype(int) +
    (df['hba']  > RO5_HBA_MAX).astype(int)
)

df = df[df['ro5_viol'] <= RO5_VIOLATIONS].copy()
report_stage('Stage 1 – Lipinski Ro5', n_before, len(df))

# Descriptor distribution summary
df[['mw', 'logp', 'hbd', 'hba']].describe().round(2)

20:24:07 [INFO] [Stage 1 – Lipinski Ro5] 645,779 → 645,538  (removed 241, kept 100.0%)


,mw,logp,hbd,hba
count,645538.00,645538.00,645538.00,645538.00
mean,353.41,2.23,1.24,4.57
std,73.41,1.13,0.83,1.54
min,85.03,-4.47,0.00,0.00
25%,305.11,1.52,1.00,3.00
50%,358.08,2.29,1.00,4.00
75%,404.08,3.00,2.00,6.00
max,678.27,7.55,7.00,12.00


## 5. Stage 2 — Veber Rules

**Rationale:** Veber et al. (2002) showed that TPSA ≤ 140 Å² AND rotatable bonds ≤ 10 are strong predictors of good oral bioavailability in rats, independent of MW. These properties reflect passive membrane permeability and metabolic stability.

In [6]:
n_before = len(df)

mask_veber = (df['tpsa'] <= VEBER_TPSA_MAX) & (df['rotb'] <= VEBER_ROTB_MAX)
df = df[mask_veber].copy()

report_stage('Stage 2 – Veber', n_before, len(df))
df[['tpsa', 'rotb']].describe().round(2)

20:24:07 [INFO] [Stage 2 – Veber] 645,538 → 627,892  (removed 17,646, kept 97.3%)


,tpsa,rotb
count,627892.00,627892.00
mean,78.44,5.26
std,23.61,2.10
min,0.00,0.00
25%,61.88,4.00
50%,77.52,5.00
75%,94.17,7.00
max,140.00,10.00


## 6. Stage 3 — Lead-like / Egan Filters

**Rationale:** The lead-like concept (Oprea, 2000) selects compounds with headroom for optimisation: MW ≤ 450, logP ≤ 4, rings ≤ 4. The Egan model (2000) captures human intestinal absorption via TPSA + logP. Compounds satisfying either criterion are retained to maximise scaffold diversity.

In [7]:
n_before = len(df)

oprea_mask = (
    (df['mw']    <= LEAD_MW_MAX) &
    (df['logp']  <= LEAD_LOGP_MAX) &
    (df['rings'] <= LEAD_RINGS_MAX)
)
egan_mask = (
    (df['tpsa'] <= EGAN_TPSA_MAX) &
    (df['logp'] <= EGAN_LOGP_MAX)
)

df = df[oprea_mask | egan_mask].copy()
report_stage('Stage 3 – Lead-like/Egan', n_before, len(df))
df[['mw', 'logp', 'rings']].describe().round(2)

20:24:07 [INFO] [Stage 3 – Lead-like/Egan] 627,892 → 624,669  (removed 3,223, kept 99.5%)


,mw,logp,rings
count,624669.00,624669.00,624669.00
mean,349.69,2.25,2.61
std,70.91,1.11,0.82
min,85.03,-4.36,0.00
25%,303.11,1.54,2.00
50%,355.11,2.31,3.00
75%,400.08,3.01,3.00
max,609.35,5.87,7.00


## 7. Stage 4 — PAINS & Structural Alerts

**Rationale:** PAINS (Pan-Assay INterference compoundS; Baell & Holloway, 2010) are substructures that produce false positives in biochemical assays via non-specific mechanisms (metal chelation, redox cycling, aggregation). The Brenk structural alert set flags reactive groups toxic in vivo. Both are encoded as RDKit `FilterCatalog` substructure queries.

This stage requires re-parsing SMILES to Mol objects — still O(n) but more expensive per compound than pure descriptor arithmetic.

In [ ]:
# Build filter catalogs once (compiled SMARTS patterns)
pains_params = FilterCatalogParams()
pains_params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS_A)
pains_params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS_B)
pains_params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS_C)
pains_catalog = FilterCatalog.FilterCatalog(pains_params)

brenk_params = FilterCatalogParams()
brenk_params.AddCatalog(FilterCatalogParams.FilterCatalogs.BRENK)
brenk_catalog = FilterCatalog.FilterCatalog(brenk_params)

n_before = len(df)

pains_flags  = []
brenk_flags  = []
pains_reasons = []
brenk_reasons = []

for smi in tqdm(df['smiles'], desc='PAINS/Brenk screening'):
    mol = smiles_to_mol(smi)
    if mol is None:
        pains_flags.append(True)   # mark as failing (shouldn't happen post-Stage 0)
        brenk_flags.append(True)
        pains_reasons.append('parse_error')
        brenk_reasons.append('parse_error')
        continue

    p_entry = pains_catalog.GetFirstMatch(mol)
    b_entry = brenk_catalog.GetFirstMatch(mol)

    pains_flags.append(p_entry is not None)
    brenk_flags.append(b_entry is not None)
    pains_reasons.append(p_entry.GetDescription() if p_entry else '')
    brenk_reasons.append(b_entry.GetDescription() if b_entry else '')

df['pains_flag']   = pains_flags
df['brenk_flag']   = brenk_flags
df['pains_reason'] = pains_reasons
df['brenk_reason'] = brenk_reasons

# Remove compounds flagged by EITHER catalog
df = df[~df['pains_flag'] & ~df['brenk_flag']].copy()

report_stage('Stage 4 – PAINS/Brenk', n_before, len(df))
print(f'  PAINS flags: {pains_flags.count(True):,}')
print(f'  Brenk flags: {brenk_flags.count(True):,}')

# Drop flag columns — no longer needed
df.drop(columns=['pains_flag', 'brenk_flag', 'pains_reason', 'brenk_reason',
                 'ro5_viol'], inplace=True)
df.to_csv(r'C:\Users\Christian\OFFTOX\data\allclean.csv', index=False)

print("Results saved to CSV!")

gc.collect()

PAINS/Brenk screening: 100%|██████████| 624669/624669 [26:53<00:00, 387.14it/s]
20:51:01 [INFO] [Stage 4 – PAINS/Brenk] 624,669 → 453,043  (removed 171,626, kept 72.5%)


  PAINS flags: 7,707
  Brenk flags: 166,720
Results saved to CSV!


0

## 8. Stage 5 — Compound Limiter ⚠️

**Before any memory-intensive 3D/ADMET computation, we cap the dataset at `COMPOUND_LIMIT` compounds.** If fewer compounds remain, all proceed. If more remain, a priority score (composite of QED-proxy descriptors) is used to select the most drug-like subset rather than random sampling.

Priority score = `QED-proxy` := normalised composite of MW (Gaussian penalty), logP (penalty), HBD, HBA, TPSA.

In [9]:
n_before = len(df)
log.info(f'Compounds entering limiter: {n_before:,} (limit = {COMPOUND_LIMIT:,})')

if n_before > COMPOUND_LIMIT:
    log.warning(
        f'{n_before:,} compounds exceeded the limiter threshold of {COMPOUND_LIMIT:,}. '
        f'Selecting top {COMPOUND_LIMIT:,} by priority score.'
    )

    # ── Priority score (higher = more drug-like) ─────────────
    # Components: proximity to ideal MW (350 Da), moderate logP,
    # low HBD+HBA counts, low TPSA penalty.
    # All terms scaled to [0, 1] via min-max normalisation.

    def minmax(s):
        rng = s.max() - s.min()
        return (s - s.min()) / rng if rng > 0 else pd.Series(0.5, index=s.index)

    # MW: penalise distance from 350 Da (drug-like sweet spot)
    mw_score   = 1 - minmax((df['mw'] - 350).abs())
    logp_score = 1 - minmax(df['logp'].clip(lower=0))  # penalise high logP
    hbd_score  = 1 - minmax(df['hbd'])
    hba_score  = 1 - minmax(df['hba'])
    tpsa_score = 1 - minmax(df['tpsa'])
    rotb_score = 1 - minmax(df['rotb'])

    df['priority'] = (
        0.25 * mw_score +
        0.20 * logp_score +
        0.15 * hbd_score +
        0.15 * hba_score +
        0.15 * tpsa_score +
        0.10 * rotb_score
    )

    df = df.nlargest(COMPOUND_LIMIT, 'priority').copy()
    df.drop(columns=['priority'], inplace=True)
    log.info(f'Limiter applied: {n_before:,} → {len(df):,}')
else:
    log.info(f'Limiter not triggered ({n_before:,} ≤ {COMPOUND_LIMIT:,}). All compounds proceed.')

gc.collect()
print(f'\n✅ Compounds proceeding to ADMET stage: {len(df):,}')

20:51:03 [INFO] Compounds entering limiter: 453,043 (limit = 10,000)
20:51:03 [WARNING] 453,043 compounds exceeded the limiter threshold of 10,000. Selecting top 10,000 by priority score.
20:51:03 [INFO] Limiter applied: 453,043 → 10,000



✅ Compounds proceeding to ADMET stage: 10,000


## 9. Stage 6 — ADMET Properties

**Rationale:** Beyond Ro5/Veber, drug candidates must demonstrate acceptable absorption, distribution, metabolism, excretion and minimal toxicity. This stage applies:

- **QED ≥ 0.3**: Quantitative Estimate of Drug-likeness (Bickerton et al., 2012, *Nat Chem* 4:90–98). Integrates 8 molecular properties using desirability functions. Range [0, 1]; approved drugs average ~0.67.
- **Fragment removal**: MW ≥ 150 Da removes solvent molecules and disconnected fragments.
- **Stereocentre count ≤ 3**: Excessive stereocentres strongly increase synthetic complexity (Lovering et al., 2009, *J Med Chem*).
- **Heavy atom count**: HA between 10–38 (equivalent to approximately 150–500 Da, fragment-to-drug space).

In [10]:
from rdkit.Chem import QED as rdQED

n_before = len(df)

qed_vals     = []
stereo_counts = []

for smi in tqdm(df['smiles'], desc='ADMET descriptors'):
    mol = smiles_to_mol(smi)
    if mol is None:
        qed_vals.append(0.0)
        stereo_counts.append(99)
        continue
    qed_vals.append(rdQED.qed(mol))
    stereo_counts.append(count_stereocentres(mol))

df['qed']    = qed_vals
df['n_stereo'] = stereo_counts

# Apply ADMET filters
mask_admet = (
    (df['qed']      >= ADMET_QED_MIN) &
    (df['mw']       >= ADMET_MW_MIN) &
    (df['n_stereo'] <= ADMET_STEREO_MAX) &
    (df['heavy']    >= 10) &
    (df['heavy']    <= 38)
)

df = df[mask_admet].copy()
report_stage('Stage 6 – ADMET', n_before, len(df))
df[['qed', 'n_stereo', 'mw', 'heavy']].describe().round(3)

ADMET descriptors: 100%|██████████| 10000/10000 [00:13<00:00, 767.47it/s]
20:51:16 [INFO] [Stage 6 – ADMET] 10,000 → 10,000  (removed 0, kept 100.0%)


,qed,n_stereo,mw,heavy
count,10000.000,10000.000,10000.000,10000.000
mean,0.799,0.035,326.281,22.439
std,0.065,0.237,31.343,2.542
min,0.460,0.000,175.111,11.000
25%,0.770,0.000,308.119,21.000
50%,0.817,0.000,332.174,23.000
75%,0.841,0.000,349.144,24.000
max,0.934,3.000,400.214,29.000


## 10. Stage 7 — 3D Conformer Generation (Optional)

**Rationale:** 3D conformer generation is a prerequisite for structure-based docking. This stage uses ETKDG (Riniker & Landrum, 2015, *J Chem Inf Model* 55:2562–2574) with UFF energy minimisation. Compounds that fail conformer embedding (typically macrocycles, highly strained systems, or unusual valence) are flagged and optionally removed.

> ⚠️ **This is the most compute-intensive stage.** ETKDG scales ~O(n·heavy²). For >500 compounds consider running on a compute node or parallelising with `multiprocessing`.

In [11]:
# Toggle: set to False to skip 3D generation
RUN_3D_CONFORMERS = True
REMOVE_CONFORMER_FAILURES = True   # Set False to keep failures (inspect manually)

if RUN_3D_CONFORMERS:
    n_before = len(df)
    embed_success = []
    conformer_sdf_records = []

    for smi in tqdm(df['smiles'], desc='ETKDG conformer embedding'):
        mol = smiles_to_mol(smi)
        if mol is None:
            embed_success.append(False)
            continue

        # Add explicit hydrogens for geometry
        mol_h = Chem.AddHs(mol)
        params = AllChem.ETKDGv3()
        params.randomSeed = 42
        params.enforceChirality = True
        params.useRandomCoords = True   # fallback for difficult geometries

        result = AllChem.EmbedMolecule(mol_h, params)

        if result == -1:
            embed_success.append(False)
        else:
            # UFF minimisation (fast, appropriate for pre-docking)
            try:
                ff_result = AllChem.UFFOptimizeMolecule(mol_h, maxIters=500)
                embed_success.append(True)
            except Exception:
                embed_success.append(True)  # keep even if minimisation fails

    df['conformer_ok'] = embed_success
    n_fail = embed_success.count(False)
    log.info(f'Conformer embedding: {embed_success.count(True):,} success, '
             f'{n_fail:,} failures')

    if REMOVE_CONFORMER_FAILURES:
        df = df[df['conformer_ok']].copy()
        report_stage('Stage 7 – 3D conformers', n_before, len(df))
    else:
        log.info('Conformer failures retained for manual inspection.')
else:
    log.info('Stage 7 (3D conformer generation) skipped.')

ETKDG conformer embedding: 100%|██████████| 10000/10000 [11:48<00:00, 14.11it/s] 
21:03:05 [INFO] Conformer embedding: 9,979 success, 21 failures
21:03:05 [INFO] [Stage 7 – 3D conformers] 10,000 → 9,979  (removed 21, kept 99.8%)


## 11. Results Summary & Export

In [12]:
print('=' * 60)
print('PIPELINE SUMMARY')
print('=' * 60)
print(f'  Input compounds:          {n_total:>10,}')
print(f'  Final compound count:     {len(df):>10,}')
print(f'  Overall retention rate:   {100*len(df)/max(n_total,1):>9.2f}%')
print('=' * 60)

# Descriptor statistics of final set
print('\nFinal compound descriptor statistics:')
summary_cols = ['mw', 'logp', 'hbd', 'hba', 'tpsa', 'rotb', 'rings', 'qed']
available = [c for c in summary_cols if c in df.columns]
display(df[available].describe().round(3))

PIPELINE SUMMARY
  Input compounds:             645,909
  Final compound count:          9,979
  Overall retention rate:        1.54%

Final compound descriptor statistics:


,mw,logp,hbd,hba,tpsa,rotb,rings,qed
count,9979.000,9979.000,9979.000,9979.000,9979.000,9979.000,9979.000,9979.000
mean,326.306,1.318,0.106,3.722,60.252,2.836,2.784,0.799
std,31.307,1.025,0.308,1.004,16.023,1.162,0.668,0.065
min,183.051,-2.710,0.000,1.000,9.230,0.000,0.000,0.460
25%,308.119,0.652,0.000,3.000,49.850,2.000,2.000,0.770
50%,332.174,1.341,0.000,4.000,60.250,3.000,3.000,0.817
75%,349.146,2.017,0.000,4.000,70.910,4.000,3.000,0.841
max,400.214,4.780,2.000,8.000,113.780,8.000,6.000,0.934


In [6]:
#if needed uncomment below
import pandas as pd
df = pd.read_csv(r"C:\Users\Christian\OFFTOX\filtered_output\filtered_compounds.csv")

In [11]:
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for notebook export
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
plot_pairs = [
    ('mw',   'Molecular Weight (Da)', 'steelblue'),
    ('logp', 'Wildman-Crippen logP',  'tomato'),
    ('tpsa', 'TPSA (Å²)',             'seagreen'),
    ('rotb', 'Rotatable Bonds',       'darkorange'),
    ('qed',  'QED Score',             'mediumpurple'),
    ('hbd',  'H-Bond Donors',         'goldenrod'),
]

for ax, (col, label, color) in zip(axes.flat, plot_pairs):
    if col in df.columns:
        ax.hist(df[col].dropna(), bins=40, color=color, edgecolor='white', linewidth=0.4)
        ax.set_xlabel(label, fontsize=10)
        ax.set_ylabel('Count', fontsize=9)
        ax.set_title(label, fontsize=11, fontweight='bold')
        ax.spines[['top', 'right']].set_visible(False)

plt.suptitle('Final Filtered Compound Library — Descriptor Distributions',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
fig_path = OUTPUT_DIR / 'descriptor_distributions.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure saved: {fig_path}')

Figure saved: filtered_output\descriptor_distributions.png


In [14]:
# Export final filtered compound list
out_csv = OUTPUT_DIR / 'filtered_compounds.csv'
df.reset_index(drop=True).to_csv(out_csv, index=False)
log.info(f'Filtered compounds saved to: {out_csv}  ({len(df):,} rows)')

# Export a SMILES-only file for downstream tools (e.g. docking, QSAR)
smiles_out = OUTPUT_DIR / 'filtered_smiles.smi'
df[['smiles', 'id']].to_csv(smiles_out, sep=' ', index=False, header=False)
log.info(f'SMILES file saved to: {smiles_out}')

print(f'\n✅ Done. {len(df):,} compounds written to {OUTPUT_DIR}/')

21:24:59 [INFO] Filtered compounds saved to: filtered_output\filtered_compounds.csv  (9,979 rows)
21:24:59 [INFO] SMILES file saved to: filtered_output\filtered_smiles.smi



✅ Done. 9,979 compounds written to filtered_output/


## Appendix: Parameter Sensitivity Reference

| Parameter | Conservative | Default (this notebook) | Permissive |
|-----------|-------------|------------------------|------------|
| Ro5 violations | 0 | 1 | 2 |
| MW max (Da) | 450 | 500 | 600 |
| logP max | 3.5 | 5.0 | 6.0 |
| TPSA max (Å²) | 90 | 140 | 160 |
| Rotatable bonds | 7 | 10 | 15 |
| QED min | 0.5 | 0.3 | 0.1 |
| Stereocentres max | 2 | 3 | 5 |

## Key References

1. **Lipinski et al. (2001)** *Adv Drug Deliv Rev* 46:3–26. Rule of Five.
2. **Veber et al. (2002)** *J Med Chem* 45:2615–2623. Oral bioavailability rules.
3. **Egan et al. (2000)** *J Med Chem* 43:3867–3877. Absorption model.
4. **Oprea (2000)** *J Comput Aided Mol Des* 14:251–264. Lead-like space.
5. **Baell & Holloway (2010)** *J Med Chem* 53:2719–2740. PAINS.
6. **Brenk et al. (2008)** *ChemMedChem* 3:435–444. Structural alerts.
7. **Bickerton et al. (2012)** *Nat Chem* 4:90–98. QED.
8. **Riniker & Landrum (2015)** *J Chem Inf Model* 55:2562–2574. ETKDG.
9. **Lovering et al. (2009)** *J Med Chem* 52:6752–6756. Escape from flatland (Fsp3/stereocentres).